# Phase 5 – Legal Statute Identification (v2)
Multilabel statute identification (100 labels, strictly macro-F1) on long Indian court judgments. Base = Phase 2-v2 model; techniques differ from v1.

**Pipeline:** data audit -> tokenise once with the phase1-v2 tokenizer (cache) -> smoke test (all code paths on 1%) -> **Stage A**: two teachers on 2 GPUs
(T1 = hand-written LoRMA + bidirectional attention + gradual top-4 layer unfreezing with layer-wise LR decay; T2 = same but causal, as ablation) ->
**Stage B**: two KD students (<40M params, 2 blocks of the base, fully trained) distilled online from T1 / T2 -> per-model dev/test logits + prediction.json +
submission.zip -> ensembles + threshold tuning (macro-F1, 2-fold CV on dev) -> final submission -> upload everything.

# Citations
1) LoRMA: Low-Rank Multiplicative Adaptation for LLMs, Bihany et. al.
2) LoRA: Low-Rank Adaptation of Large Language Models, Hu et. al.
3) On Greedy Approaches to Hierarchical Aggregation, Porter et. al.
4) Class Weighting Technique to Deal with Imbalanced Class Problem in Machine Learning: Methodological Research
5) AdaLoRA: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning, Zhang et. al.
6) Layer freezing / unfreezing strategies (INT, BOT, TOP, ADT-L, ADT-H)
7) Distilling the Knowledge in a Neural Network, Hinton et. al.
8) LLMs for LLMs: A Structured Prompting Methodology for Long Legal Documents Klem et. al.


In [ ]:
!find /kaggle/input -name model.pt

In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
import time; T_NB0 = time.time()
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool
import regex
!pip install -q peft scikit-learn pandas
!pip uninstall -y torchao

In [ ]:
HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os, json, ijson, hashlib
import numpy as np
from collections import Counter
from huggingface_hub import hf_hub_download

REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {"train": "Phase_5_LSI/Train/train.json",
         "dev":   "Phase_5_LSI/Train/dev.json",
         "test":  "Phase_5_LSI/Test/test.json"}
paths = {k: hf_hub_download(REPO, f, repo_type="dataset", token=HF_TOKEN) for k, f in FILES.items()}

def audit(path, name):
    n = 0; keys = Counter(); lens = []; ncl = []; lab_cnt = Counter()
    ids = []; hashes = Counter(); id_types = Counter(); empty_lab = 0
    with open(path, "rb") as f:
        for r in ijson.items(f, "item"):
            n += 1; keys.update(r.keys()); ids.append(r["id"]); id_types[type(r["id"]).__name__] += 1
            t = r.get("text") or ""; lens.append(len(t))
            hashes[hashlib.md5(t.encode("utf-8", "ignore")).hexdigest()] += 1
            if "labels" in r:
                L = [int(x) for x in r["labels"]]
                ncl.append(len(L)); lab_cnt.update(L); empty_lab += (len(L) == 0)
    print(f"\n{'='*60}\n{name}: {n} records | fields: {dict(keys)} | id types: {dict(id_types)}")
    print("sample ids:", ids[:3], "| duplicate ids:", n - len(set(ids)), "| duplicate texts:", n - len(hashes))
    print("chars p50/p90/p99/max:", [int(np.percentile(lens, q)) for q in (50, 90, 99, 100)])
    if ncl:
        c = np.array(sorted(lab_cnt.values()))
        print(f"distinct labels: {len(lab_cnt)} | min/max label id: {min(lab_cnt)}/{max(lab_cnt)}")
        print(f"labels per doc mean/median/max: {np.mean(ncl):.2f}/{np.median(ncl):.0f}/{max(ncl)} | docs with 0 labels: {empty_lab}")
        print(f"label doc-frequency: median {np.median(c):.0f} | labels with <=5 docs: {(c<=5).sum()} | <=20: {(c<=20).sum()} | >=500: {(c>=500).sum()}")
        print("top 10 labels:", lab_cnt.most_common(10))
    return set(ids), set(lab_cnt), lab_cnt

tr_ids, tr_labs, _ = audit(paths["train"], "TRAIN")
dv_ids, dv_labs, _ = audit(paths["dev"],   "DEV")
te_ids, _, _       = audit(paths["test"],  "TEST")

print("\nid overlap train/dev:", len(tr_ids & dv_ids), "| train/test:", len(tr_ids & te_ids), "| dev/test:", len(dv_ids & te_ids))
print("dev labels unseen in train:", len(dv_labs - tr_labs), "| train labels absent in dev:", len(tr_labs - dv_labs))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")



TRAIN_FILES = ["Phase_5_LSI/Train/train.json", "Phase_5_LSI/Train/dev.json"]

TEST_FILES = ["Phase_5_LSI/Test/test.json",]

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)


HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase2-v1"

OUTPUT_DIR = "/kaggle/working/cs781_phase5" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer_v2")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data_v3")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 2 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_paths = []
    for tf in TEST_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            test_paths.append(p); print(f"Downloaded Test File: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
    return local_train_paths, test_paths

In [ ]:
train_paths, test_paths = download_data()

# inspect_json_files(train_paths, n=2)

In [ ]:
P = {os.path.basename(p): p for p in train_paths}
TEST = test_paths
print(P.keys(), TEST)

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["vocab.json","merges.json","tokenizer_config.json"]:
    hf_hub_download("DumbLad/legal-bpe-tokenizer-phase1-v2", f, repo_type="model",
                    local_dir="/kaggle/working/tok_phase1_v2", token=HF_TOKEN)

## Stage 0 – extra data checks (decide preprocessing / imbalance handling)

## Code files (written once, imported by every job)

In [ ]:
%%writefile lora_layers.py
import math, torch, torch.nn as nn, torch.nn.functional as F

class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B(A(dropout(x))).  W0 frozen, A (r x in) kaiming init, B (out x r) zero init."""
    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        assert isinstance(base, nn.Linear) and base.bias is None
        self.base, self.r, self.scale = base, r, alpha / r
        for p in base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

class LoRMALinear(nn.Module):
    """LoRMA (multiplicative low-rank adaptation): W' = (I + s*B A) W0  ->  y = h + s*B(A(h)),  h = W0 x.
       A (r x out) kaiming, B (out x r) zero => W' = W0 at init. The identity term is the rank-inflation: W' stays
       full-rank although B A has rank <= r. Cost = two thin matmuls on the OUTPUT of the frozen layer."""
    def __init__(self, base: nn.Linear, r=16, alpha=None, dropout=0.0):
        super().__init__()
        assert isinstance(base, nn.Linear) and base.bias is None
        self.base, self.r, self.scale = base, r, (alpha or r) / r
        for p in base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale

TARGETS = ("wq", "wk", "wv", "wo", "w_gate", "w_up", "w_down")

def apply_lora(model, r=16, alpha=None, dropout=0.05, targets=TARGETS, kind="lora"):
    for p in model.parameters(): p.requires_grad = False
    for mod in list(model.modules()):                       # snapshot: new LoRA modules are not revisited
        for name, ch in list(mod.named_children()):
            if name in targets and isinstance(ch, nn.Linear):
                setattr(mod, name, LoRMALinear(ch, r, alpha, 0.0) if kind == "lorma" else LoRALinear(ch, r, alpha or 2 * r, dropout))
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{kind.upper()} attached: r={r}, trainable adapter params = {n/1e6:.2f}M", flush=True)
    return model

def apply_lorma(model, r=32, **kw): return apply_lora(model, r, kind="lorma", **kw)


In [ ]:
%%writefile cjpe_data.py
import os, re, json
import numpy as np
from multiprocessing import Pool

MAX_CHARS = 80000          # keep first 40k + last 40k chars of very long documents
DECISION_WORDS = ["dismissed", "allowed", "acquitted", "convicted", "upheld", "affirmed", "restored",
                  "quashed", "remanded", "guilty", "liable", "rejected", "granted", "partly", "decreed", "sustained"]

def clean(s, pre):
    s = s.replace("\u00a0", " ")
    if len(s) > MAX_CHARS:
        s = s[: MAX_CHARS // 2] + " " + s[-MAX_CHARS // 2:]
    if pre != "raw":
        s = s.lower()
        s = re.sub(r"\bno\.\s*(?=\d)", "number ", s)
    if pre == "mask":
        s = re.sub(r"\b\d{1,2}[-./]\d{1,2}[-./]\d{2,4}\b", " <date> ", s)
        s = re.sub(r"\d{3,}", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()

_TOK = None
def _init(tok_dir):
    global _TOK
    from bpe_tokenizer import PureByteLevelBPETokenizer
    _TOK = PureByteLevelBPETokenizer.load(tok_dir)

def _enc(a):
    text, pre = a
    return np.asarray(_TOK.encode(clean(text, pre)), dtype=np.uint16)

def build_cache(src, name, pre, tok_dir, out_dir, nproc=None, force=False, block=512, n_labels=100):
    """Stream a JSON array of {id,text,labels}; NO dedup (every doc, incl. every test doc, must keep a row);
       tokenise in parallel; save ragged ids + multi-hot label matrix Y (N x n_labels)."""
    import ijson
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{name}_{pre}.npz"
    if os.path.exists(path) and not force:
        print("cache exists:", path); return path
    nproc = nproc or max(1, os.cpu_count() or 2)
    ids_l, off, rows, dids = [], [0], [], []

    def flush(buf, pool):
        res = pool.map(_enc, [(t, pre) for t, _, _ in buf], chunksize=4)
        for (t, labs, d), a in zip(buf, res):
            ids_l.append(a); off.append(off[-1] + len(a)); rows.append(labs); dids.append(d)

    with Pool(nproc, initializer=_init, initargs=(tok_dir,)) as pool, open(src, "rb") as f:
        buf = []
        for rec in ijson.items(f, "item"):
            labs = [int(x) for x in (rec.get("labels") or [])]
            buf.append((rec["text"], labs, str(rec["id"])))
            if len(buf) >= block:
                flush(buf, pool); buf = []
        if buf: flush(buf, pool)
    Y = np.zeros((len(rows), n_labels), dtype=np.uint8)
    for i, labs in enumerate(rows):
        if labs: Y[i, labs] = 1
    np.savez(path, ids=np.concatenate(ids_l), off=np.asarray(off, dtype=np.int64), Y=Y, did=np.asarray(dids))
    print(f"{name}/{pre}: {len(rows)} docs, {off[-1]/1e6:.1f}M tokens, avg labels/doc {Y.sum(1).mean():.2f} -> {path}")
    return path

def load_cache(path):
    z = np.load(path)
    return {k: z[k] for k in z.files}

def doc_ids(C, i):
    return C["ids"][C["off"][i]:C["off"][i + 1]]

def key_ids(tok):
    return np.array(sorted({tok.encode(" " + w)[0] for w in DECISION_WORDS}), dtype=np.int64)

# ------------------------------------------------------------------ long-document representation
def chunk_starts(n, body, overlap):
    if n <= body: return [0]
    stride = max(1, body - overlap)
    s = list(range(0, n - body + 1, stride))
    if s[-1] + body < n: s.append(n - body)
    return s

def n_chunks(n, c):
    if c["view"] != "chunks": return 1
    return min(c["max_chunks"], len(chunk_starts(n, c["chunk_len"] - 2, c["overlap"])))

def make_chunks(ids, c, bos, eos, keys=None):
    body = c["chunk_len"] - 2; n = len(ids); v = c["view"]
    if v == "head":   parts = [ids[:body]]
    elif v == "tail": parts = [ids[-body:]]
    elif v == "headtail":
        h = body // 4
        parts = [ids if n <= body else np.concatenate([ids[:h], ids[n - (body - h):]])]
    else:
        parts = [ids[s:s + body] for s in chunk_starts(n, body, c["overlap"])]
        M = c["max_chunks"]
        if len(parts) > M:
            m, pk = len(parts), c["pick"]
            if pk == "first": sel = list(range(M))
            elif pk == "ht":
                h = M // 4; sel = list(range(h)) + list(range(m - (M - h), m))
            elif pk == "even": sel = sorted(set(np.linspace(0, m - 1, M).round().astype(int).tolist()))
            elif pk == "kw":
                sc = np.array([np.isin(p, keys).mean() for p in parts])
                top = [int(i) for i in np.argsort(-sc) if i not in (0, m - 1)][:M - 2]
                sel = sorted({0, m - 1, *top})
            else: raise ValueError(pk)
            parts = [parts[i] for i in sel]
    return [np.concatenate([[bos], p.astype(np.int64), [eos]]) for p in parts]

def make_batches(C, idx, c, shuffle, rng, budget):
    """budget = max #chunks per micro-batch. If c['tok_budget'] is set, also cap (#chunks x padded length) and
       sort docs by length inside a window -> almost no padding, and every batch has a similar memory footprint."""
    off = C["off"]; body = c["chunk_len"] - 2; tb = c.get("tok_budget")
    nch = {i: n_chunks(int(off[i + 1] - off[i]), c) for i in idx}
    clen = {i: min(int(off[i + 1] - off[i]), body) + 2 for i in idx}      # padded length of the doc's chunks
    order = list(idx)
    if shuffle: rng.shuffle(order)
    W = 512 if tb else 256
    key = (lambda j: (clen[j], nch[j])) if tb else (lambda j: nch[j])
    out = []
    for s in range(0, len(order), W):
        cur, tot = [], 0
        for j in sorted(order[s:s + W], key=key):
            if cur and (tot + nch[j] > budget or (tb and (tot + nch[j]) * clen[j] > tb)):
                out.append(cur); cur, tot = [], 0
            cur.append(j); tot += nch[j]
        if cur: out.append(cur)
    if shuffle: rng.shuffle(out)
    return out

def subset(C, frac, seed):
    """Fixed random subsample (multi-label: no single-label stratification)."""
    n = len(C["did"])
    if frac >= 1: return list(range(n))
    rng = np.random.RandomState(seed)
    return sorted(rng.choice(n, max(1, int(round(frac * n))), replace=False).tolist())

In [ ]:
%%writefile lsi.py
import os, json, math, time, random, argparse, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.cluster import KMeans
from llama_scratch import Config, LlamaLM, PRESETS, apply_rope, repeat_kv
from bpe_tokenizer import PureByteLevelBPETokenizer
from lora_layers import apply_lora, apply_lorma
from tqdm import tqdm
from cjpe_data import load_cache, doc_ids, make_chunks, make_batches, subset, key_ids

NL = 100

# ---------------------------------------------------------------- base model (Phase 1 checkpoint)
def load_base(ckpt, keep=None):
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    sd = next((ck[k] for k in ("model", "model_state_dict", "state_dict") if isinstance(ck.get(k), dict)), ck)
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    cfg = ck.get("config") or ck.get("cfg")
    if isinstance(cfg, dict): cfg = Config(**cfg)
    if cfg is None:
        V, D = sd["tok_emb.weight"].shape
        L = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("layers."))
        nh = next(p["n_heads"] for p in PRESETS.values() if p["d_model"] == D)
        nkv = sd["layers.0.attn.wk.weight"].shape[0] // (D // nh)
        cfg = Config(vocab_size=V, d_model=D, n_layers=L, n_heads=nh, n_kv_heads=nkv, ffn_hidden=sd["layers.0.mlp.w_gate.weight"].shape[0])
    cfg.attn_impl = "sdpa"
    if keep:                                                   # student: keep only these layers (renumbered 0..k-1)
        idx = {old: i for i, old in enumerate(keep)}; new = {}
        for k, v in sd.items():
            if k.startswith("layers."):
                _, n, rest = k.split(".", 2)
                if int(n) in idx: new[f"layers.{idx[int(n)]}.{rest}"] = v
            else: new[k] = v
        sd = new; cfg.n_layers = len(keep)
    m = LlamaLM(cfg)
    res = m.load_state_dict(sd, strict=False)
    bad = [k for k in res.missing_keys if "lm_head" not in k]
    assert not bad and not res.unexpected_keys, (bad[:5], res.unexpected_keys[:5])
    return m

# ---------------------------------------------------------------- label hierarchy: statute "families"
def make_groups(Y, k, seed=0):
    Y = Y.astype(np.float32); co = Y.T @ Y
    d = np.sqrt(np.clip(np.diag(co), 1, None)); cos = co / np.outer(d, d)
    lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(cos).labels_
    return [int(x) for x in lab]

# ---------------------------------------------------------------- document model
class LSIModel(nn.Module):
    """LM (+hand-written LoRA) encodes each chunk causally -> chunk vector -> pooling across chunks (mean/attn/hier)
       -> shared trunk -> 3 heads: statutes (100), statute family (K), #statutes. Final statute logit = fine + gmix * family logit."""
    def __init__(self, base, c):
        super().__init__()
        base.lm_head = nn.Identity()                                  # never used here; keeps the param count honest
        self.lm = apply_lora(base, c["r"]) if c["peft"] == "lora" else apply_lorma(base, c["r"]) if c["peft"] == "lorma" else base
        self.bidir = c.get("bidir", False)
        D = base.cfg.d_model; self.agg, self.ctype, self.gc = c["agg"], c["ctype"], c.get("grad_ckpt", False)
        K = max(c["groups"]) + 1
        self.register_buffer("gid", torch.tensor(c["groups"], dtype=torch.long), persistent=False)
        self.attn_q = nn.Linear(D, 1) if self.agg in ("attn", "hier") else None
        self.pos = self.enc = None
        if self.agg == "hier":
            self.pos = nn.Embedding(64, D)
            self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(D, 8, 2 * D, 0.1, activation="gelu", batch_first=True, norm_first=True),
                                             1, enable_nested_tensor=False)
        self.proj = nn.Sequential(nn.Dropout(.1), nn.Linear(D, D), nn.GELU(), nn.Dropout(.1))
        self.cls, self.grp, self.cnt = nn.Linear(D, NL), nn.Linear(D, K), nn.Linear(D, 1)
        self.gmix = nn.Parameter(torch.tensor(0.5))

    def core(self): return self.lm

    def head_params(self):
        ps = [p for m in (self.proj, self.cls, self.grp, self.cnt) for p in m.parameters()] + [self.gmix]
        for m in (self.attn_q, self.pos, self.enc):
            if m is not None: ps += list(m.parameters())
        return ps

    def bi_mask(self, lens, T, dtype):
        """additive mask (N, nh|1, T, T): pad keys hidden; ALiBi uses the symmetric distance |i-j| (no causal cut)."""
        b = self.core(); kv = torch.arange(T, device=lens.device)[None] < lens[:, None]            # (N, T) valid keys
        m = (~kv)[:, None, None, :].float() * -1e4
        if b.alibi_slopes is not None:
            ar = torch.arange(T, device=lens.device)
            m = m - b.alibi_slopes.view(1, -1, 1, 1).to(lens.device) * (ar[:, None] - ar[None, :]).abs().float()[None, None]
        return m.to(dtype)

    @staticmethod
    def bi_layer(l, x, cos, sin, m):
        a = l.attn; B, T, _ = x.shape; h = l.attn_norm(x)
        q = a.wq(h).view(B, T, a.nh, a.hd).transpose(1, 2)
        k = a.wk(h).view(B, T, a.nkv, a.hd).transpose(1, 2); v = a.wv(h).view(B, T, a.nkv, a.hd).transpose(1, 2)
        if cos is not None: q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        o = F.scaled_dot_product_attention(q, repeat_kv(k, a.n_rep), repeat_kv(v, a.n_rep), attn_mask=m.to(q.dtype))
        x = x + a.wo(o.transpose(1, 2).reshape(B, T, -1))
        return x + l.mlp(l.mlp_norm(x))

    def hidden(self, idx, lens=None):
        b = self.core(); x = b.tok_emb(idx); T = idx.size(1)
        cos = sin = None
        if b.cfg.pos_enc == "rope": cos, sin = b.rope_cos[:T], b.rope_sin[:T]
        if self.bidir:                                                   # full (non-causal) attention inside each chunk
            m = self.bi_mask(lens, T, torch.float16 if torch.is_autocast_enabled() else x.dtype)
            for l in b.layers:
                if self.gc and self.training and torch.is_grad_enabled():
                    x = checkpoint(lambda y, l=l: self.bi_layer(l, y, cos, sin, m), x, use_reentrant=False)
                else: x = self.bi_layer(l, x, cos, sin, m)
            return b.norm(x)
        for l in b.layers:
            if self.gc and self.training and torch.is_grad_enabled():
                x = checkpoint(lambda y, l=l: l(y, cos, sin, b.alibi_slopes, None, None, False)[0], x, use_reentrant=False)
            else:
                x, _ = l(x, cos, sin, b.alibi_slopes, None, None, False)
        return b.norm(x)

    def forward(self, ids, lens, owner, nb, aux=False):
        h = self.hidden(ids, lens); N, T, D = h.shape
        if self.ctype == "last": v = h[torch.arange(N, device=h.device), lens - 1]
        else:
            m = (torch.arange(T, device=h.device)[None] < lens[:, None]).unsqueeze(-1)
            v = (h.float() * m).sum(1) / lens[:, None]
        v = v.float()
        counts = torch.bincount(owner, minlength=nb); C = int(counts.max())
        start = torch.cumsum(counts, 0) - counts
        pos = torch.arange(N, device=h.device) - start[owner]
        G = v.new_zeros(nb, C, D).index_put((owner, pos), v)                      # (docs, chunks, D)
        ar = torch.arange(C, device=h.device)[None]
        M = ar < counts[:, None]; Mf = M.unsqueeze(-1).to(G.dtype)
        if self.agg == "mean": r = (G * Mf).sum(1) / counts[:, None].to(G.dtype)
        else:
            if self.agg == "hier":
                assert C <= 64
                G = self.enc(G + self.pos(ar.expand(nb, C)), src_key_padding_mask=~M)
            a = self.attn_q(G).squeeze(-1).float().masked_fill(~M, -1e4).softmax(-1)
            r = (a.unsqueeze(-1).to(G.dtype) * G).sum(1)
        t = self.proj(r.float())
        zg = self.grp(t).float()
        z = self.cls(t).float() + self.gmix * zg[:, self.gid]
        if not aux: return z
        return z, zg, self.cnt(t).squeeze(-1).float(), t

# ---------------------------------------------------------------- multi-task weighted loss
def make_loss(Ytr, groups, c, dev):
    n = len(Ytr); pos = Ytr.sum(0).clip(min=1).astype(np.float64)
    pw = torch.tensor(((n - pos) / pos) ** c["pw_pow"], dtype=torch.float32).clamp(1.0, c["pw_max"]).to(dev)
    K = max(groups) + 1
    Gm = torch.zeros(NL, K); Gm[torch.arange(NL), torch.tensor(groups)] = 1; Gm = Gm.to(dev)
    cnt = Ytr.sum(1); mu, sd = float(cnt.mean()), float(cnt.std() + 1e-6); w = c["task_w"]
    def f(z, zg, zc, y):
        l_main = F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)                 # rare statutes up-weighted
        l_grp = F.binary_cross_entropy_with_logits(zg, ((y @ Gm) > 0).float())           # statute-family task
        l_cnt = F.smooth_l1_loss(zc, (y.sum(1) - mu) / sd)                               # #statutes task
        return w["main"] * l_main + w["group"] * l_grp + w["count"] * l_cnt
    return f

# ---------------------------------------------------------------- metrics / thresholds (macro-F1)
TS = np.arange(0.05, 0.951, 0.025)

def to_pred(P, thr):
    pred = P > thr
    rows = np.where(~pred.any(1))[0]
    if len(rows): pred[rows, (P[rows] - thr).argmax(1)] = True                          # every doc gets >= 1 statute
    return pred

def f1s(P, Y, thr):
    pred = to_pred(P, thr); y = Y.astype(bool)
    tp = (pred & y).sum(0).astype(float); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0)
    macro = float(np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1)))
    micro = float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1))
    return dict(macro_f1=macro, micro_f1=micro)

def f1_grid(P, Y, ts=TS):
    pred = P[:, :, None] > ts[None, None, :]; y = Y.astype(bool)[:, :, None]
    tp = (pred & y).sum(0).astype(float); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)                                      # (labels, thresholds)

def tune_global(P, Y, ts=TS): return float(ts[f1_grid(P, Y, ts).mean(0).argmax()])

def tune_per_label(P, Y, g, ts=TS, lam=0.5):
    best = ts[f1_grid(P, Y, ts).argmax(1)]
    return lam * best + (1 - lam) * g                                                    # shrink toward the global threshold

# ---------------------------------------------------------------- batching + prediction
def collate(C, batch, c, pad, BOS, EOS, keys):
    chunks, owner = [], []
    for b, i in enumerate(batch):
        for p in make_chunks(doc_ids(C, i), c, BOS, EOS, keys):
            chunks.append(p); owner.append(b)
    T = (max(len(p) for p in chunks) + 7) // 8 * 8
    ids = torch.full((len(chunks), T), pad, dtype=torch.long); lens = torch.zeros(len(chunks), dtype=torch.long)
    for k, p in enumerate(chunks):
        ids[k, :len(p)] = torch.from_numpy(p); lens[k] = len(p)
    y = torch.from_numpy(C["Y"][batch].astype(np.float32))
    return ids, lens, torch.tensor(owner), y

@torch.no_grad()
def predict(model, C, idx, ec, pad, BOS, EOS, keys):
    model.eval(); Z, K = [], []
    for b in tqdm(make_batches(C, idx, dict(ec, tok_budget=ec.get("tok_budget_eval")), False, None, ec.get("chunk_bs_eval", 16)), desc="eval", mininterval=30, leave=False):
        ids, lens, owner, _ = collate(C, b, ec, pad, BOS, EOS, keys)
        with torch.autocast("cuda", dtype=torch.float16):
            z = model(ids.cuda(), lens.cuda(), owner.cuda(), len(b))
        Z.append(z.float().cpu()); K += b
    Z = torch.cat(Z); pos = {k: i for i, k in enumerate(K)}
    return Z[[pos[i] for i in idx]]

# ---------------------------------------------------------------- knowledge distillation (multilabel)
def kd_loss(zs, zt, ts, tt, k):
    """soft-label BCE on temperature-scaled logits (x T^2) + MSE between the pooled trunk vectors (same width)."""
    T = k["T"]
    l_soft = F.binary_cross_entropy_with_logits(zs / T, torch.sigmoid(zt / T)) * T * T
    return k["alpha"] * l_soft + k["beta"] * F.mse_loss(ts, tt)

# ---------------------------------------------------------------- per-model submission files
def write_submission(P, dids, thr, d):
    os.makedirs(d, exist_ok=True); pred = to_pred(P, thr)
    sub = [{"id": i, "pred": [int(j) for j in np.where(r)[0]]} for i, r in zip(dids, pred)]
    json.dump(sub, open(f"{d}/prediction.json", "w"))
    with zipfile.ZipFile(f"{d}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{d}/prediction.json", "prediction.json")

def load_lsi(path, device="cuda"):                                   # re-open any saved model: m, cfg = load_lsi(".../name.pt")
    ck = torch.load(path, map_location="cpu", weights_only=False); c = ck["cfg"]
    m = LSIModel(load_base(c["ckpt"], c.get("keep_layers")), c); m.load_state_dict(ck["state"], strict=False)
    return m.to(device).eval(), c

# ---------------------------------------------------------------- one experiment
def run(c):
    t0 = time.time(); dev = torch.device("cuda")
    random.seed(c["seed"]); np.random.seed(c["seed"]); torch.manual_seed(c["seed"]); rng = random.Random(c["seed"])
    tok = PureByteLevelBPETokenizer.load(c["tok_dir"]); pad, BOS, EOS = tok.pad_token_id, tok.bos_token_id, tok.eos_token_id
    keys = key_ids(tok) if c["pick"] == "kw" else None
    Ctr = load_cache(f"{c['cache']}/train_{c['pre']}.npz"); Cdv = load_cache(f"{c['cache']}/dev_{c['pre']}.npz")
    tr, dv = subset(Ctr, c["frac"], c["seed"]), subset(Cdv, c["dev_frac"], 0)
    ec = dict(c); Ytr, Ydv = Ctr["Y"][tr], Cdv["Y"][dv]
    c["groups"] = make_groups(Ctr["Y"], c["n_groups"], 0)
    nbat = len(make_batches(Ctr, tr, c, True, random.Random(0), c["chunk_bs"]))
    steps = max(1, nbat // c["accum"]) * c["epochs"]
    model = LSIModel(load_base(c["ckpt"], c.get("keep_layers")), c).to(dev)
    n_total = sum(p.numel() for p in model.parameters())
    if c.get("max_params"): assert n_total <= c["max_params"], f"student has {n_total/1e6:.1f}M params > {c['max_params']/1e6:.0f}M"
    print(f'[{c["name"]}] total params {n_total/1e6:.2f}M | peft {c["peft"]} | bidir {c.get("bidir", False)}', flush=True)
    core = model.core()
    if c["peft"] == "full":
        for p in core.parameters(): p.requires_grad = True
    prior = np.clip(Ytr.mean(0), 1e-4, 1 - 1e-4)                                          # start the head at the label prior
    model.cls.bias.data = torch.tensor(np.log(prior / (1 - prior)), dtype=torch.float32, device=dev)
    head = model.head_params(); hid = {id(p) for p in head}
    ad = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]            # adapters (LoRMA A,B) or all core params
    # ---- layer unfreezing: top-N blocks' base weights (+ final norm) train with layer-wise LR decay
    unf, nL, gradual = [], len(core.layers), c.get("unfreeze_mode", "static") == "gradual"
    for j in range(min(c.get("unfreeze_n", 0) if c["peft"] != "full" else 0, nL)):
        ps = [p for n, p in core.layers[nL - 1 - j].named_parameters() if not n.endswith((".A", ".B"))]
        if j == 0: ps += list(core.norm.parameters())
        for p in ps: p.requires_grad = not gradual
        unf.append(ps)
    groups = [dict(params=ad, lr=c["lr"], weight_decay=0.0), dict(params=head, lr=c["head_lr"], weight_decay=0.01)]
    groups += [dict(params=ps, lr=c.get("lr_u", 3e-5) * c.get("llrd", 0.8) ** j, weight_decay=0.01) for j, ps in enumerate(unf)]
    opt = torch.optim.AdamW(groups)
    opt_ids = {id(p) for g in groups for p in g["params"]}
    saved = {n: p for n, p in model.named_parameters() if id(p) in opt_ids}                 # everything the optimizer may touch
    n_on = [0]
    def unfreeze_step(pg):
        if not gradual: return
        k = min(len(unf), int(len(unf) * pg / c.get("unfreeze_by", 0.5)))
        while n_on[0] < k:
            for p in unf[n_on[0]]: p.requires_grad = True
            print(f'[{c["name"]}] unfroze block {nL - 1 - n_on[0]} at progress {pg:.2f}', flush=True); n_on[0] += 1
    trainable = [p for g in opt.param_groups for p in g["params"]]
    base_lrs = [g["lr"] for g in opt.param_groups]; warm = max(1, int(.05 * steps))
    teacher = None
    if c.get("kd"):                                                                          # online KD: frozen teacher sees the same chunks
        tk = torch.load(c["kd"]["teacher"], map_location="cpu", weights_only=False); tc = tk["cfg"]
        for f in ("chunk_len", "overlap", "max_chunks", "pick", "view", "pre"): assert tc[f] == c[f], f"teacher/student differ in {f}"
        teacher = LSIModel(load_base(tc["ckpt"], tc.get("keep_layers")), tc); teacher.load_state_dict(tk["state"], strict=False)
        teacher = teacher.to(dev).eval()
        for p in teacher.parameters(): p.requires_grad = False
        print(f'[{c["name"]}] KD from {os.path.basename(c["kd"]["teacher"])} (teacher dev F1 {tk.get("dev_f1")})', flush=True)
    lossf = make_loss(Ytr, c["groups"], c, dev); scaler = torch.amp.GradScaler()
    os.makedirs(c["out"], exist_ok=True); ck_path = f"{c['out']}/{c['name']}.pt"
    limit = c["max_minutes"] * 60; n_ev = c["n_evals"]
    S = dict(best=-1.0, state=None, s=None, step=0, t_eval=0.0); hist = []
    step = micro = tok_count = ev_done = 0; tt = time.time(); done = False
    print(f'[{c["name"]}] train docs {len(tr)} | nominal steps {steps} | time box {c["max_minutes"]} min', flush=True)

    def prog(st): return max(st / steps, (time.time() - tt - S["t_eval"]) / limit)

    def do_eval(st):
        te = time.time()
        Z = predict(model, Cdv, dv, ec, pad, BOS, EOS, keys); P = torch.sigmoid(Z).numpy()
        g = tune_global(P, Ydv); s = dict(f1s(P, Ydv, g), thr=g)
        print(f'[{c["name"]}] EVAL step {st}: {s}', flush=True)
        if s["macro_f1"] > S["best"]:
            S.update(best=s["macro_f1"], s=s, step=st, state={n: p.detach().cpu().clone() for n, p in saved.items()})
            torch.save(dict(cfg=c, state=S["state"], step=st, dev_f1=s["macro_f1"]), ck_path)
        S["t_eval"] += time.time() - te; model.train()

    vsub = sorted(np.random.RandomState(1).choice(dv, min(len(dv), c.get("val_docs", 400)), replace=False).tolist())
    Yv = Cdv["Y"][vsub]

    def quick_val(st):                                   # cheap monitor on a fixed dev subset; its time is NOT charged to the train box
        te = time.time()
        Z = predict(model, Cdv, vsub, ec, pad, BOS, EOS, keys); P = torch.sigmoid(Z).numpy()
        vl = F.binary_cross_entropy_with_logits(Z, torch.from_numpy(Yv.astype(np.float32))).item()
        tl = float(np.mean([h[1] for h in hist[-10:]])) if hist else float("nan")
        print(f'[{c["name"]}] VAL step {st}: val-loss {vl:.4f} | train-loss (avg last 10 logs) {tl:.4f} | '
              f'macro-F1@0.3 {f1s(P, Yv, 0.3)["macro_f1"]:.4f}', flush=True)
        S["t_eval"] += time.time() - te; model.train()


    for ep in range(c["epochs"]):
        model.train()
        for b in make_batches(Ctr, tr, c, True, rng, c["chunk_bs"]):
            ids, lens, owner, y = collate(Ctr, b, c, pad, BOS, EOS, keys)
            with torch.autocast("cuda", dtype=torch.float16):
                z, zg, zc, tv = model(ids.to(dev), lens.to(dev), owner.to(dev), len(b), aux=True)
                if teacher is not None:
                    with torch.no_grad(): zt, _, _, tt_ = teacher(ids.to(dev), lens.to(dev), owner.to(dev), len(b), aux=True)
            loss = lossf(z, zg, zc, y.to(dev))
            if teacher is not None:
                loss = (1 - c["kd"]["alpha"]) * loss + kd_loss(z, zt.float(), tv.float(), tt_.float(), c["kd"])
            scaler.scale(loss / c["accum"]).backward(); micro += 1; tok_count += int(lens.sum())
            if micro % c["accum"]: continue
            pg = prog(step)
            sc = step / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, pg))))
            for g_, l0 in zip(opt.param_groups, base_lrs): g_["lr"] = l0 * sc
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); step += 1; opt.zero_grad(set_to_none=True)
            unfreeze_step(prog(step))
            if step % 10 == 0: hist.append((step, float(loss.detach())))
            pg = prog(step)
            if step % 25 == 0:
                el = time.time() - tt - S["t_eval"]
                eta = el * (1 - pg) / max(pg, 1e-6)
                print(f'[{c["name"]}] step {step}/{steps} prog {pg:.2f} loss {loss.item():.4f} elapsed {el/60:.1f}m '
                      f'train-ETA {eta/60:.1f}m {tok_count/max(el,1):.0f} real-tok/s', flush=True)
            if c.get("val_every") and step % c["val_every"] == 0 and pg < 1.0: quick_val(step)
            if pg >= (ev_done + 1) / n_ev:
                do_eval(step); ev_done = min(n_ev, int(pg * n_ev))
            if pg >= 1.0: done = True; break
        if done: break
    if ev_done < n_ev: do_eval(step)
    train_min = (time.time() - tt - S["t_eval"]) / 60
    model.load_state_dict(S["state"], strict=False)
    json.dump(hist, open(f"{c['out']}/{c['name']}_loss.json", "w"))

    if c.get("final_infer", True):                                                        # full dev + test logits, on this GPU
        os.makedirs(f"{c['out']}/logits", exist_ok=True)
        Cte = load_cache(f"{c['cache']}/test_{c['pre']}.npz")
        for split, C in (("dev", Cdv), ("test", Cte)):
            idx = list(range(len(C["did"])))[: (c.get("infer_limit") or None)]
            Z = predict(model, C, idx, ec, pad, BOS, EOS, keys); dids = [str(x) for x in C["did"][idx]]
            torch.save(dict(Z=Z, did=dids, Y=C["Y"][idx]), f"{c['out']}/logits/{c['name']}_{split}.pt")
            print(f'[{c["name"]}] saved {split} logits ({len(idx)} docs)', flush=True)
            if split == "dev": Pd, Yd_ = torch.sigmoid(Z).numpy(), C["Y"][idx]; thr0 = tune_global(Pd, Yd_); full_dev = dict(f1s(Pd, Yd_, thr0), thr=thr0)
            else: write_submission(torch.sigmoid(Z).numpy(), dids, thr0, f"{c['out']}/submissions/{c['name']}")   # this model's own submission
        print(f'[{c["name"]}] full-dev {full_dev} -> {c["out"]}/submissions/{c["name"]}/', flush=True)
    return dict(name=c["name"], cfg=c, macro_f1=S["best"], dev_sel=S["s"], best_step=S["step"], steps_done=step, steps_nominal=steps,
                full_dev=locals().get("full_dev"), total_params=n_total, trainable_adapter_params=sum(p.numel() for p in ad),
                unfrozen_base_params=sum(p.numel() for ps in unf for p in ps), peak_mem_gb=torch.cuda.max_memory_allocated() / 1e9,
                minutes=(time.time() - t0) / 60, train_minutes=train_min, tokens_per_sec=tok_count / max(1.0, train_min * 60))

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--jobs"); a = ap.parse_args()
    for c in json.load(open(a.jobs)):
        res = f"{c['out']}/results.jsonl"
        if os.path.exists(res) and any(json.loads(l)["name"] == c["name"] for l in open(res)): continue
        try: r = run(c)
        except Exception:
            import traceback, gc; print(f"[{c['name']}] FAILED", flush=True); traceback.print_exc(); gc.collect(); torch.cuda.empty_cache(); continue
        with open(res, "a") as f: f.write(json.dumps(r) + "\n")
        torch.cuda.empty_cache()

## Tokenise once and cache (preprocessing `norm`)  – every later experiment only re-chunks cached ids

In [ ]:
import sys, json, ijson, torch
sys.path.insert(0, ".")
from llama_scratch import Config, LlamaLM
from bpe_tokenizer import PureByteLevelBPETokenizer
from cjpe_data import clean
tok = PureByteLevelBPETokenizer.load("/kaggle/working/tok_phase1_v2")
ck = torch.load("/kaggle/input/datasets/koooshiieee/phase2v2/model.pt", map_location="cpu", weights_only=False)
cfg = Config(**ck["cfg"]); cfg.attn_impl = "sdpa"
lm = LlamaLM(cfg); lm.load_state_dict(ck["model"]); lm = lm.cuda().eval()
with open(P["dev.json"], "rb") as f:
    texts = [d["text"] for _, d in zip(range(80), ijson.items(f, "item"))]
for pre in ("raw", "norm"):
    nll = ntok = nchar = 0
    for t in texts:
        ids = tok.encode(clean(t, pre))[:1000]
        if len(ids) < 50: continue
        x = torch.tensor([[tok.bos_token_id] + ids], device="cuda")
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            loss = lm(x[:, :-1], targets=x[:, 1:]).item()
        n = x.size(1) - 1; nll += loss * n; ntok += n; nchar += len(tok.decode(ids))
    print(f"{pre:5s} loss/token {nll/ntok:.3f} | loss/char {nll/nchar:.3f} | tokens/char {ntok/nchar:.3f}")
del lm; torch.cuda.empty_cache()

In [ ]:
import sys, time, os, json, numpy as np
sys.path.insert(0, ".")
CKPT_PATH = "/kaggle/input/datasets/koooshiieee/phase2v2/model.pt"
TOK_DIR   = "/kaggle/working/tok_phase1_v2"
WORK      = "/kaggle/working"
CACHE     = f"{WORK}/tok_cache5_v2"
OUT       = f"{WORK}/lsi_v2"
os.makedirs(OUT, exist_ok=True)
from cjpe_data import build_cache
SRC = {"train": P["train.json"], "dev": P["dev.json"], "test": TEST[0]}
t0 = time.time()
for split, path in SRC.items(): build_cache(path, split, "raw", TOK_DIR, CACHE)
print(f"tokenised in {(time.time()-t0)/60:.1f} min")

## Job launcher (2 x T4: jobs are split across both GPUs, finished jobs are skipped on re-run)

In [ ]:
import json, copy, subprocess, time
import pandas as pd
BASE = dict(ckpt=CKPT_PATH, tok_dir=TOK_DIR, cache=CACHE, out=OUT,
            peft="lorma", r=32, lr=3e-4, head_lr=6e-4, pre="raw",                  # hand-written LoRMA on every linear of the LM
            bidir=True, ctype="mean",                                                # non-causal attention inside each chunk + masked-mean pooling
            unfreeze_n=4, unfreeze_mode="gradual", unfreeze_by=0.5, lr_u=3e-5, llrd=0.8,   # top-4 blocks' base weights, one block per 12.5% of progress, LR x0.8 per block down
            view="chunks", chunk_len=1024, overlap=128, max_chunks=4, pick="ht",
            agg="hier", head="mlp",
            n_groups=10, task_w=dict(main=1.0, group=0.3, count=0.1), pw_pow=0.5, pw_max=8.0,
            chunk_bs=48, tok_budget=12288, chunk_bs_eval=128, tok_budget_eval=32768, accum=2, epochs=1, frac=1.0, dev_frac=0.15,
            n_evals=3, max_minutes=80, seed=67, grad_ckpt=False, final_infer=True,
            kd=None, keep_layers=None, max_params=None,
            val_every=150, val_docs=400)   # every 150 optimizer steps: val-loss + macro-F1 on 400 fixed dev docs (0/None = off)
# KD student: Phase-2-v2 embeddings + 2 of its 12 blocks (5, 11), ALL weights trained, attention pooling -> ~38.6M params (< 40M, asserted)
STU = dict(peft="full", keep_layers=[5, 11], agg="attn", lr=1e-4, head_lr=6e-4, unfreeze_n=0, max_params=40_000_000)
KD = dict(T=2.0, alpha=0.5, beta=0.5)          # loss = (1-alpha)*multi-task loss + alpha*soft-BCE(T) + beta*MSE(trunk vectors)
T1, T2 = "lsi2_T1_lorma_bidir_unf", "lsi2_T2_lorma_causal_unf"
S1, S2 = "lsi2_S1_kd_bidir", "lsi2_S2_kd_causal"

def J(name, base, **kw):
    d = copy.deepcopy(base); d.update(kw); d["name"] = name; return d

def results(out=None):
    p = f"{out or OUT}/results.jsonl"; return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

def run_jobs(jobs, out=None):
    out = out or jobs[0]["out"]
    done = {r["name"] for r in results(out)}; jobs = [j for j in jobs if j["name"] not in done]
    if not jobs: print("nothing to run"); return
    names = {j["name"] for j in jobs}; procs = []; t0 = time.time()
    for g in range(2):
        mine = jobs[g::2]
        if not mine: continue
        jf = f"{out}/jobs_{g}.json"; json.dump(mine, open(jf, "w"))
        procs.append(subprocess.Popen(["python", "-u", "lsi.py", "--jobs", jf], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"),
                                      stdout=open(f"{out}/log_{g}.txt", "a"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        n = len([r for r in results(out) if r["name"] in names])
        print(f"{time.strftime('%H:%M:%S')} | {n}/{len(jobs)} jobs done | {(time.time()-t0)/60:.0f} min elapsed")
        for g in range(len(procs)):
            try: print(f"   GPU{g}:", open(f"{out}/log_{g}.txt").read().strip().splitlines()[-1][:200])
            except Exception: pass
    missing = names - {r["name"] for r in results(out)}
    if missing: print("FAILED / MISSING (see log_*.txt):", sorted(missing))

## Smoke test (run ONCE, ~15-25 min; checks every code path on 1% of the data, then delete it)

In [ ]:
import shutil
# SMOKE = the real run in miniature: same jobs, same names, same stages, same ensemble/submission cell - only data/time/inference are cut.
SM_OUT = f"{WORK}/lsi_smoke"; shutil.rmtree(SM_OUT, ignore_errors=True); os.makedirs(SM_OUT, exist_ok=True)
SM_KW = dict(out=SM_OUT, frac=0.02, dev_frac=0.02, n_evals=3, max_minutes=5, infer_limit=128, val_every=10, val_docs=64)
SM_T, SM_S = dict(BASE, **SM_KW), dict(BASE, **STU, **SM_KW)
# Stage A
run_jobs([J(T1, SM_T), J(T2, SM_T, bidir=False, ctype="last", seed=7)])
# Stage B
run_jobs([J(S1, SM_S, kd=dict(KD, teacher=f"{SM_OUT}/{T1}.pt")),
          J(S2, SM_S, bidir=False, ctype="last", seed=7, kd=dict(KD, teacher=f"{SM_OUT}/{T2}.pt"))])
for r in results(SM_OUT): print(r["name"], r["dev_sel"], f'{r["total_params"]/1e6:.1f}M params', f'{r["tokens_per_sec"]:.0f} tok/s', f'peak {r["peak_mem_gb"]:.1f} GB', f'{r["train_minutes"]:.1f} min')

In [ ]:
!tail -n 40 /kaggle/working/lsi_smoke/log_0.txt
!tail -n 40 /kaggle/working/lsi_smoke/log_1.txt

In [ ]:
# point the ensemble/submission cell (further down) at the smoke outputs
OUT_REAL = f"{WORK}/lsi_v2"; OUT = SM_OUT; N_EXPECT = 128

In [ ]:
OUT = f"{WORK}/lsi_v2"; N_EXPECT = 13019                          # undo the smoke redirect
TOTAL_MIN = 210
T_NB0 = time.time() - 8 * 60   # smoke test + setup ~8 min; counts from launch, not from notebook start
n = {s: np.diff(np.load(f"{CACHE}/{s}_raw.npz")["off"]) for s in ("train", "dev", "test")}
cap = BASE["max_chunks"] * (BASE["chunk_len"] - 2)
tr_tok = np.minimum(n["train"], cap).sum(); inf_tok = sum(np.minimum(n[s], cap).sum() for s in ("dev", "test"))
inf_T = inf_tok / 25_000 / 60; inf_S = inf_T * 0.25                       # teachers (12 blocks) and students (2 blocks) infer on both GPUs in parallel
used = (time.time() - T_NB0) / 60
rem = TOTAL_MIN - used - inf_T - inf_S - 45                                # 30 = evals + loads + thresholds + safety
MIN_A = int(max(15, min(90, 0.6 * rem))); MIN_S = int(max(10, min(55, rem - MIN_A)))
BASE["max_minutes"] = MIN_A; BASE_S = dict(BASE, **STU, max_minutes=MIN_S)
print(f"used {used:.0f} min | inference ~ {inf_T:.0f}+{inf_S:.0f} min | teacher train box {MIN_A} min | student (KD) box {MIN_S} min | "
      f"one teacher epoch ~ {tr_tok/14500/60:.0f} min at 14.5k tok/s -> ~{min(1, MIN_A/(tr_tok/14500/60)):.2f} epochs")


In [ ]:
# kill leftover lsi.py jobs from an interrupted run, then show GPU state
import subprocess
subprocess.run(["pkill", "-f", "lsi.py --jobs"])
time.sleep(5)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
print(open(f"{OUT}/log_0.txt").read()[-1500:] if os.path.exists(f"{OUT}/log_0.txt") else "no log_0 yet")

In [ ]:
# STAGE A - teachers, one per GPU, in parallel. Each is trained end to end; it saves {name}.pt, logits and its own submission folder.
run_jobs([J(T1, BASE),                                                   # LoRMA + bidirectional attention + gradual top-4 unfreezing
          J(T2, BASE, bidir=False, ctype="last", seed=7)])               # control: same recipe but causal attention (ablation + ensemble member)
for r in results(): print(r["name"], r["dev_sel"], f'steps {r["steps_done"]}/{r["steps_nominal"]}', f'{r["train_minutes"]:.0f} min')


In [ ]:
# STAGE B - knowledge distillation: each student (<40M params) learns online from the frozen teacher of its own flavour.
assert all(os.path.exists(f"{OUT}/{t}.pt") for t in (T1, T2)), "a teacher is missing - re-run stage A"
run_jobs([J(S1, BASE_S, kd=dict(KD, teacher=f"{OUT}/{T1}.pt")),
          J(S2, BASE_S, bidir=False, ctype="last", seed=7, kd=dict(KD, teacher=f"{OUT}/{T2}.pt"))])
for r in results(): print(r["name"], r["dev_sel"], f'{r["total_params"]/1e6:.1f}M params', f'{r["train_minutes"]:.0f} min')

In [ ]:
import importlib, lsi, zipfile, shutil
importlib.reload(lsi)
LOG = f"{OUT}/logits"
NAMES = [n for n in (T1, T2, S1, S2) if os.path.exists(f"{LOG}/{n}_test.pt")]          # only models that finished
TEACH = [n for n in NAMES if n in (T1, T2)]
L = {n: {s: torch.load(f"{LOG}/{n}_{s}.pt", weights_only=False) for s in ("dev", "test")} for n in NAMES}
assert all(L[n]["dev"]["did"] == L[NAMES[0]]["dev"]["did"] and L[n]["test"]["did"] == L[NAMES[0]]["test"]["did"] for n in NAMES)
Yd, did_t = L[NAMES[0]]["dev"]["Y"], L[NAMES[0]]["test"]["did"]
sig = lambda Z: torch.sigmoid(Z).numpy()
cands = {n: (sig(L[n]["dev"]["Z"]), sig(L[n]["test"]["Z"])) for n in NAMES}
if len(TEACH) > 1: cands["ens_teachers"] = tuple(np.mean([cands[n][i] for n in TEACH], 0) for i in (0, 1))
if len(NAMES) > 2: cands["ens_all"] = tuple(np.mean([cands[n][i] for n in NAMES], 0) for i in (0, 1))

def cv_eval(P, Y, seed=0):                       # 2-fold on dev: tune on one half, score the other half
    perm = np.random.RandomState(seed).permutation(len(Y)); a, b = perm[:len(Y)//2], perm[len(Y)//2:]
    out = dict(global_=[], per_label=[])
    for i, j in ((a, b), (b, a)):
        g = lsi.tune_global(P[i], Y[i])
        out["global_"].append(lsi.f1s(P[j], Y[j], g)["macro_f1"])
        out["per_label"].append(lsi.f1s(P[j], Y[j], lsi.tune_per_label(P[i], Y[i], g))["macro_f1"])
    return {k: float(np.mean(v)) for k, v in out.items()}

rows = [dict(cand=k, mode=m, cv_macro_f1=v) for k, (Pd, _) in cands.items() for m, v in cv_eval(Pd, Yd).items()]
tbl = pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False).round(4); print(tbl)

# a prediction.json + submission.zip for EVERY model and ensemble (threshold mode = that candidate's best CV mode)
def thr_for(k):
    Pd = cands[k][0]; g = lsi.tune_global(Pd, Yd); m = tbl[tbl.cand == k].iloc[0]["mode"]
    return g if m == "global_" else lsi.tune_per_label(Pd, Yd, g)
for k, (Pd, Pt) in cands.items(): lsi.write_submission(Pt, did_t, thr_for(k), f"{OUT}/submissions/{k}")
print("written:", sorted(os.listdir(f"{OUT}/submissions")))

best = tbl.iloc[0]["cand"]; Pd, Pt = cands[best]; thr = thr_for(best)
print("chosen:", best, tbl.iloc[0]["mode"], "| dev macro-F1 (tuned on dev, optimistic):", lsi.f1s(Pd, Yd, thr))
for f in ("prediction.json", "submission.zip"): shutil.copy(f"{OUT}/submissions/{best}/{f}", f"{OUT}/{f}")   # final submission = best candidate

# validation
from cjpe_data import load_cache
with zipfile.ZipFile(f"{OUT}/submission.zip") as z: assert z.namelist() == ["prediction.json"]; chk = json.loads(z.read("prediction.json"))
test_ids = [str(x) for x in load_cache(f"{CACHE}/test_raw.npz")["did"]]
N_EXPECT = globals().get("N_EXPECT", 13019)                      # 128 in smoke mode (infer_limit), 13019 for the real run
assert [o["id"] for o in chk] == test_ids[:N_EXPECT] and len(chk) == N_EXPECT
assert all(o["pred"] and len(set(o["pred"])) == len(o["pred"]) and all(isinstance(x, int) and 0 <= x <= 99 for x in o["pred"]) for o in chk)
print("submission valid:", len(chk), "docs | avg statutes/doc", np.mean([len(o["pred"]) for o in chk]).round(2), "(train avg 3.71)")


In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN); REPO = f"{api.whoami()['name']}/legal-lsi-phase5-v2"
api.create_repo(REPO, repo_type="model", private=True, exist_ok=True)
up = lambda p, dst: api.upload_file(path_or_fileobj=p, path_in_repo=dst, repo_id=REPO, repo_type="model")
for n in NAMES: up(f"{OUT}/{n}.pt", f"models/{n}.pt")                                    # every trained model (LoRMA/unfrozen/head weights; students = full weights)
api.upload_folder(folder_path=f"{OUT}/submissions", repo_id=REPO, repo_type="model", path_in_repo="submissions")   # prediction.json + submission.zip of each model / ensemble
for f in ("results.jsonl", "prediction.json", "submission.zip"): up(f"{OUT}/{f}", f)    # last two = the final chosen submission
for f in ("lsi.py", "cjpe_data.py", "lora_layers.py", "llama_scratch.py", "bpe_tokenizer.py"): up(f, f"code/{f}")
print("Uploaded to https://huggingface.co/" + REPO)


In [ ]:
import re, matplotlib.pyplot as plt
pat = re.compile(r"\[(\S+)\] VAL step (\d+): val-loss ([\d.]+) \| train-loss \(avg last 10 logs\) ([\d.]+)")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for g in (0, 1):
    for name, st, vl, tl in pat.findall(open(f"{OUT}/log_{g}.txt").read()):
        pass
data = {}
for g in (0, 1):
    for name, st, vl, tl in pat.findall(open(f"{OUT}/log_{g}.txt").read()):
        data.setdefault(name, []).append((int(st), float(vl), float(tl)))
for name, rows in data.items():
    s, v, t = zip(*rows)
    ax[0].plot(s, t, label=name); ax[1].plot(s, v, label=name)
ax[0].set_title("train loss"); ax[1].set_title("val loss (400 dev docs)")
for a in ax: a.set_xlabel("step"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(LOSS_PLOT_PATH, dpi=150); plt.show()

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
DATA_REPO = f"{api.whoami()['name']}/legal-lsi-phase5-v2-data"
api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=CACHE, repo_id=DATA_REPO, repo_type="dataset", path_in_repo="tok_cache5_v2")        # token ids + multi-hot labels
if os.path.isdir(f"{OUT}/logits"):
    api.upload_folder(folder_path=f"{OUT}/logits", repo_id=DATA_REPO, repo_type="dataset", path_in_repo="logits")   # dev/test logits of both models
for f in ("log_0.txt", "log_1.txt", "results.jsonl", "prediction.json", "submission.zip"):
    if os.path.exists(f"{OUT}/{f}"):
        api.upload_file(path_or_fileobj=f"{OUT}/{f}", path_in_repo=f"run/{f}", repo_id=DATA_REPO, repo_type="dataset")
print("https://huggingface.co/datasets/" + DATA_REPO)